# Aula 04 — Funções de ativação

Este tutorial deriva e implementa **sigmoid, tanh, ReLU, Leaky ReLU e softplus** somente
com NumPy. Mediremos faixa, inclinação, saturação, simetrias e estabilidade numérica.
O foco é a operação local $a=\phi(z)$; ainda não construiremos cache nem backward da rede.

**Dependências mínimas:** Python 3.11, NumPy 1.26, Matplotlib 3.8 e nbformat 5.9.
Não há download, credencial ou dado externo.


## Goal

Ao terminar, você deverá conseguir:

1. implementar ativações vetorizadas que preservam shape;
2. derivar suas inclinações locais;
3. explicar saturação e regiões de gradiente zero;
4. distinguir ponto não diferenciável de derivada numericamente pequena;
5. calcular sigmoid e softplus sem overflow evitável;
6. verificar derivadas por diferenças centrais fora das quinas;
7. escolher ativação oculta e de saída de acordo com o papel matemático.


## Setup

- arrays em `float64` para verificações numéricas;
- `SEED = 20260909` para amostras sintéticas;
- convenção da ReLU em $z=0$: derivada implementada igual a zero;
- `alpha = 0.01` na Leaky ReLU;
- diferenças centrais apenas em pontos diferenciáveis;
- warnings da fórmula ingênua são suprimidos localmente porque o overflow é a
  contraprova esperada; nenhuma implementação recomendada gera warning.


In [ ]:
from __future__ import annotations

import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
ALPHA = 0.01
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=8, suppress=True)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
    "alpha_leaky_relu": ALPHA,
})


## Steps

### 1. Implementações estáveis

Para $z\ge0$, a forma usual da sigmoid é segura. Para $z<0$, reescrevemos a fração
usando $e^z$, que não explode. Softplus usa `logaddexp(0, z)`, equivalente a
$\log(1+e^z)$, mas estável para valores extremos.


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    positive = z >= 0.0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)
    return out


def tanh(z: np.ndarray) -> np.ndarray:
    return np.tanh(np.asarray(z, dtype=np.float64))


def relu(z: np.ndarray) -> np.ndarray:
    return np.maximum(np.asarray(z, dtype=np.float64), 0.0)


def leaky_relu(z: np.ndarray, alpha: float = ALPHA) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    assert alpha >= 0.0
    return np.where(z >= 0.0, z, alpha * z)


def softplus(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    return np.logaddexp(0.0, z)


### 2. Derivadas locais

Usaremos a própria saída para calcular sigmoid e tanh:
$\sigma'(z)=\sigma(z)(1-\sigma(z))$ e
$\tanh'(z)=1-\tanh^2(z)$. ReLU e Leaky ReLU são definidas por regiões.
Softplus tem uma conexão útil: sua derivada é a sigmoid.


In [ ]:
def d_sigmoid(z: np.ndarray) -> np.ndarray:
    s = sigmoid(z)
    return s * (1.0 - s)


def d_tanh(z: np.ndarray) -> np.ndarray:
    t = tanh(z)
    return 1.0 - t**2


def d_relu(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    return (z > 0.0).astype(np.float64)  # convenção: phi'(0) = 0


def d_leaky_relu(z: np.ndarray, alpha: float = ALPHA) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    return np.where(z >= 0.0, 1.0, alpha)


def d_softplus(z: np.ndarray) -> np.ndarray:
    return sigmoid(z)


### 3. Faixas, shapes e valores de referência

Toda ativação elemento a elemento preserva o shape. Os pontos $-2$, $0$ e $2$ tornam
as diferenças de faixa visíveis e permitem conferir valores conhecidos.


In [ ]:
z_reference = np.array([[-2.0, 0.0, 2.0]])
reference = {
    "sigmoid": sigmoid(z_reference),
    "tanh": tanh(z_reference),
    "relu": relu(z_reference),
    "leaky_relu": leaky_relu(z_reference),
    "softplus": softplus(z_reference),
}

for name, values in reference.items():
    assert values.shape == z_reference.shape
    assert np.isfinite(values).all()
    print(f"{name:>11}: {values}")

assert np.isclose(sigmoid(np.array([0.0]))[0], 0.5)
assert np.isclose(tanh(np.array([0.0]))[0], 0.0)
assert np.isclose(softplus(np.array([0.0]))[0], np.log(2.0))


### 4. Estabilidade nos extremos

Na fórmula ingênua `1 / (1 + exp(-z))`, $z=-1000$ exige calcular $e^{1000}$ e provoca
overflow intermediário. O resultado final pode até virar zero, mas o caminho numérico é
frágil. A forma por ramos evita esse intermediário. A mesma ideia vale para softplus.


In [ ]:
z_extreme = np.array([-1000.0, -100.0, 0.0, 100.0, 1000.0])

with np.errstate(over="ignore"):
    naive_exp = np.exp(-z_extreme)
    sigmoid_naive = 1.0 / (1.0 + naive_exp)
    softplus_naive = np.log1p(np.exp(z_extreme))

sigmoid_safe = sigmoid(z_extreme)
softplus_safe = softplus(z_extreme)

assert np.isinf(naive_exp[0])
assert np.isinf(softplus_naive[-1])
assert np.isfinite(sigmoid_safe).all()
assert np.isfinite(softplus_safe).all()
assert sigmoid_safe[0] == 0.0 and sigmoid_safe[-1] == 1.0
assert np.isclose(softplus_safe[-1], 1000.0)
print({
    "sigmoid_estavel": sigmoid_safe.tolist(),
    "softplus_estavel": softplus_safe.tolist(),
    "naive_sigmoid_teve_intermediario_infinito": bool(np.isinf(naive_exp).any()),
    "naive_softplus_teve_saida_infinita": bool(np.isinf(softplus_naive).any()),
})


### 5. Verificação numérica das derivadas

Para função escalar $f$, a diferença central é
$(f(z+h)-f(z-h))/(2h)$. Usamos $h=10^{-6}$ e excluímos $z=0$ das funções com quina.
O erro absoluto máximo deve permanecer abaixo de $10^{-8}$.


In [ ]:
def central_difference(function, z: np.ndarray, h: float = 1e-6) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    return (function(z + h) - function(z - h)) / (2.0 * h)


smooth_points = np.array([-4.0, -1.3, -0.2, 0.4, 1.7, 4.0])
piecewise_points = np.array([-4.0, -1.3, -0.2, 0.4, 1.7, 4.0])
gradient_checks = {}

for name, function, derivative, points in (
    ("sigmoid", sigmoid, d_sigmoid, smooth_points),
    ("tanh", tanh, d_tanh, smooth_points),
    ("relu", relu, d_relu, piecewise_points),
    ("leaky_relu", leaky_relu, d_leaky_relu, piecewise_points),
    ("softplus", softplus, d_softplus, smooth_points),
):
    numeric = central_difference(function, points)
    analytic = derivative(points)
    error = float(np.max(np.abs(numeric - analytic)))
    gradient_checks[name] = error
    assert error < 1e-8

print(gradient_checks)


### 6. O que acontece exatamente em zero?

ReLU é contínua em zero, mas as inclinações laterais são diferentes: zero à esquerda e
um à direita. A diferença central retorna $0.5$, enquanto nossa implementação escolhe
zero. Não é falha do gradient check: a derivada clássica simplesmente não existe ali.


In [ ]:
h = 1e-6
relu_left_slope = float(((relu(np.array([0.0])) - relu(np.array([-h]))) / h)[0])
relu_right_slope = float(((relu(np.array([h])) - relu(np.array([0.0]))) / h)[0])
relu_central_at_zero = float(central_difference(relu, np.array([0.0]), h)[0])
relu_convention_at_zero = float(d_relu(np.array([0.0]))[0])

assert relu_left_slope == 0.0
assert relu_right_slope == 1.0
assert np.isclose(relu_central_at_zero, 0.5)
assert relu_convention_at_zero == 0.0
print({
    "inclinacao_esquerda": relu_left_slope,
    "inclinacao_direita": relu_right_slope,
    "diferenca_central": relu_central_at_zero,
    "convencao_implementada": relu_convention_at_zero,
})


### 7. Saturação e produto de derivadas

Sigmoid tem inclinação máxima $1/4$; tanh, máxima 1. Nos extremos, ambas se aproximam
de constantes e suas derivadas se aproximam de zero. Em uma cadeia idealizada de 20
sigmoids avaliadas em zero, o produto máximo das inclinações é $(1/4)^{20}$.


In [ ]:
z_saturation = np.array([-20.0, -5.0, 0.0, 5.0, 20.0])
saturation_table = np.column_stack([
    z_saturation,
    sigmoid(z_saturation),
    d_sigmoid(z_saturation),
    tanh(z_saturation),
    d_tanh(z_saturation),
])

max_sigmoid_chain_20 = float(0.25**20)
assert np.isclose(d_sigmoid(np.array([0.0]))[0], 0.25)
assert np.isclose(d_tanh(np.array([0.0]))[0], 1.0)
assert max_sigmoid_chain_20 < 1e-12
print("colunas: z, sigmoid, d_sigmoid, tanh, d_tanh")
print(saturation_table)
print({"produto_maximo_20_sigmoids": max_sigmoid_chain_20})


### 8. Simetrias e centralização

Sigmoid satisfaz $\sigma(-z)=1-\sigma(z)$; tanh é ímpar. Para uma entrada simétrica,
a média da sigmoid fica perto de $0.5$, enquanto tanh fica perto de zero. ReLU remove
o lado negativo e desloca a média para valores positivos.


In [ ]:
z_symmetric = rng.normal(size=200_000)
symmetry_grid = np.linspace(-20.0, 20.0, 2001)

sigmoid_symmetry_error = float(np.max(np.abs(sigmoid(-symmetry_grid) - (1.0 - sigmoid(symmetry_grid)))))
tanh_symmetry_error = float(np.max(np.abs(tanh(-symmetry_grid) + tanh(symmetry_grid))))
means = {
    "entrada": float(z_symmetric.mean()),
    "sigmoid": float(sigmoid(z_symmetric).mean()),
    "tanh": float(tanh(z_symmetric).mean()),
    "relu": float(relu(z_symmetric).mean()),
    "leaky_relu": float(leaky_relu(z_symmetric).mean()),
}

assert sigmoid_symmetry_error < 2e-16
assert tanh_symmetry_error < 2e-16
assert abs(means["sigmoid"] - 0.5) < 0.002
assert abs(means["tanh"]) < 0.005
print({
    "erro_simetria_sigmoid": sigmoid_symmetry_error,
    "erro_simetria_tanh": tanh_symmetry_error,
    "medias": means,
})


### 9. ReLU inativa e Leaky ReLU

Com pré-ativações deslocadas para a região negativa, ReLU produz muitos zeros e sua
derivada local também é zero nesses pontos. Leaky ReLU conserva inclinação `alpha` no
lado negativo. Isso não prova que ela sempre treina melhor; apenas mede o mecanismo.


In [ ]:
z_shifted = rng.normal(loc=-2.0, scale=1.0, size=100_000)
relu_zero_fraction = float(np.mean(relu(z_shifted) == 0.0))
relu_zero_derivative_fraction = float(np.mean(d_relu(z_shifted) == 0.0))
leaky_zero_derivative_fraction = float(np.mean(d_leaky_relu(z_shifted) == 0.0))
leaky_min_slope = float(d_leaky_relu(z_shifted).min())

assert np.isclose(relu_zero_fraction, relu_zero_derivative_fraction)
assert relu_zero_fraction > 0.97
assert leaky_zero_derivative_fraction == 0.0
assert np.isclose(leaky_min_slope, ALPHA)
print({
    "fracao_relu_zero": relu_zero_fraction,
    "fracao_derivada_relu_zero": relu_zero_derivative_fraction,
    "fracao_derivada_leaky_zero": leaky_zero_derivative_fraction,
    "menor_inclinacao_leaky": leaky_min_slope,
})


### 10. Curvas das funções e derivadas

Os eixos são compartilhados para facilitar comparação. Linhas verticais marcam zero;
as legendas informam função e derivada. A figura é evidência visual, não substitui as
verificações numéricas anteriores.


In [ ]:
z_plot = np.linspace(-6.0, 6.0, 1201)
functions = [
    ("Sigmoid", sigmoid, d_sigmoid),
    ("tanh", tanh, d_tanh),
    ("ReLU", relu, d_relu),
    (f"Leaky ReLU (alpha={ALPHA})", leaky_relu, d_leaky_relu),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (name, function, derivative) in zip(axes.ravel(), functions):
    ax.plot(z_plot, function(z_plot), label=r"$\phi(z)$", linewidth=2)
    ax.plot(z_plot, derivative(z_plot), label=r"$\phi'(z)$", linestyle="--")
    ax.axvline(0.0, color="black", linewidth=0.8, alpha=0.5)
    ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.3)
    ax.set_title(name)
    ax.set_xlabel("pré-ativação z")
    ax.set_ylabel("valor")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle("Funções de ativação e inclinações locais")
fig.tight_layout()
plt.show()


## Checks

Consolidamos propriedades matemáticas e numéricas. Elas cobrem faixa, simetria,
estabilidade, derivadas e o caso não diferenciável; não avaliam treinamento.


In [ ]:
monotonic_grid = np.linspace(-50.0, 50.0, 10_001)
checks = {
    "sigmoid_finita_extremos": bool(np.isfinite(sigmoid_safe).all()),
    "softplus_finita_extremos": bool(np.isfinite(softplus_safe).all()),
    "sigmoid_intervalo": bool(np.all((sigmoid_safe >= 0.0) & (sigmoid_safe <= 1.0))),
    "tanh_intervalo": bool(np.all(np.abs(tanh(monotonic_grid)) <= 1.0)),
    "sigmoid_monotona": bool(np.all(np.diff(sigmoid(monotonic_grid)) >= 0.0)),
    "tanh_impar": tanh_symmetry_error < 2e-16,
    "sigmoid_simetrica": sigmoid_symmetry_error < 2e-16,
    "derivadas_verificadas": max(gradient_checks.values()) < 1e-8,
    "relu_nao_diferenciavel_zero": relu_left_slope != relu_right_slope,
    "softplus_deriva_sigmoid": float(np.max(np.abs(d_softplus(smooth_points) - sigmoid(smooth_points)))) == 0.0,
    "leaky_mantem_inclinacao": np.isclose(leaky_min_slope, ALPHA),
}
assert all(checks.values())
print(checks)
print(f"{sum(checks.values())}/{len(checks)} contratos satisfeitos")


## Limites do experimento

- Saturação foi medida localmente; ainda não propagamos gradientes por uma rede treinada.
- A fração de ReLUs inativas depende da distribuição artificial escolhida.
- O produto $(1/4)^{20}$ é um limite ilustrativo, não a norma real do gradiente de uma MLP.
- Não comparamos tempo de treino, generalização ou inicializações.
- A convenção da derivada na quina precisa ser documentada, mas um ponto isolado raramente
  determina sozinho a dinâmica de lotes contínuos.


## Next Steps

Na **Aula 05 — Forward pass vetorizado**, combinaremos camada densa e ativação em uma
interface consistente, registrando o cache mínimo de cada intermediário para preparar o
backward manual posterior.
